# 03 — Returns and Discount Factor
**Week 3 | RL Fundamentals**

The **return** G_t is the cumulative reward from time t onwards:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

The discount factor γ ∈ [0,1) controls how much the agent cares about future rewards.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Computing Returns

In [ ]:
def compute_returns(rewards, gamma):
    """Compute discounted returns for a list of rewards (backwards pass)."""
    G = np.zeros(len(rewards))
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        G[t] = running
    return G

# Example trajectory: -0.1 per step, +10 at end
rewards = [-0.1] * 10 + [10.0]
t = np.arange(len(rewards))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, gamma in zip(axes, [0.5, 0.9, 0.99]):
    G = compute_returns(rewards, gamma)
    ax.bar(t, G, color='steelblue', edgecolor='white')
    ax.set_title(f'γ = {gamma}\nG_0 = {G[0]:.2f}')
    ax.set_xlabel('Time step'); ax.set_ylabel('Return G_t')
plt.suptitle('Return at each timestep for different γ', y=1.02)
plt.tight_layout(); plt.show()

## 2. Effect of γ on Discount Weights

In [ ]:
steps = np.arange(0, 50)
fig, ax = plt.subplots(figsize=(8, 3.5))
for gamma, color in [(0.5,'tomato'),(0.9,'steelblue'),(0.99,'seagreen'),(1.0,'darkorange')]:
    weights = gamma ** steps
    ax.plot(steps, weights, label=f'γ={gamma}', linewidth=2, color=color)
ax.set_xlabel('Steps into the future k')
ax.set_ylabel('Discount weight γ^k')
ax.set_title('How much do future rewards count?')
ax.legend(); plt.tight_layout(); plt.show()
print("\nA reward 20 steps away is worth:")
for g in [0.5, 0.9, 0.99, 1.0]:
    print(f"  γ={g}: {g**20:.4f} of its face value")

## 3. Episodic vs Continuing Tasks

In [ ]:
# Episodic: trajectory has a natural end (e.g. reaching the goal)
def episodic_returns(n_steps=15, goal_reward=10.0, step_cost=-0.1, gamma=0.99):
    rewards = [step_cost] * (n_steps-1) + [goal_reward]
    return compute_returns(rewards, gamma)

# Continuing: no terminal state; must use discounting to keep G finite
def continuing_returns(n_steps=50, reward_per_step=1.0, gamma=0.95):
    rewards = [reward_per_step] * n_steps
    return compute_returns(rewards, gamma)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
g_ep = episodic_returns()
axes[0].bar(range(len(g_ep)), g_ep, color='steelblue', edgecolor='white')
axes[0].set_title('Episodic Task (γ=0.99)')
axes[0].set_xlabel('Step t'); axes[0].set_ylabel('Return G_t')

g_cont = continuing_returns()
axes[1].bar(range(len(g_cont)), g_cont, color='seagreen', edgecolor='white')
axes[1].set_title('Continuing Task (γ=0.95)')
axes[1].set_xlabel('Step t'); axes[1].set_ylabel('Return G_t')
plt.tight_layout(); plt.show()

print(f"Continuing task: geometric series limit = R/(1-γ) = {1.0/(1-0.95):.1f}")
print(f"Empirical G_0 = {g_cont[0]:.2f}")

## 4. The Bellman Equation for Returns
G_t = R_{t+1} + γ * G_{t+1}  — the key recursive structure used by all RL algorithms.

In [ ]:
rewards = np.random.randn(20)  # random rewards
gamma   = 0.9
G = compute_returns(rewards, gamma)

# Verify Bellman equation: G[t] == rewards[t+1] + gamma * G[t+1]
for t in range(len(G)-1):
    lhs = G[t]
    rhs = rewards[t] + gamma * G[t+1]
    assert np.isclose(lhs, rhs), f"Mismatch at t={t}: {lhs:.4f} vs {rhs:.4f}"
print("✅ Bellman equation verified for all timesteps")

## ✅ Exercises
1. Set γ=0 and γ=1 and describe what kind of agent each creates. Is γ=1 ever safe for a continuing task?
2. Compute the optimal γ for a task where rewards are [1, 1, 1, ..., 100] after 50 steps. At what γ does the far reward contribute at least 50% of G_0?
3. **Challenge**: implement `compute_returns_forward` — compute G_t without reversing the array (hint: use the geometric series formula).

## ✅ Exercise Solutions

### Exercise 1 — γ = 0 and γ = 1

- **γ = 0**: The agent is **completely myopic** — it only cares about the *immediate* next reward. `G_t = R_{t+1}` always.
- **γ = 1**: The agent is **far-sighted** — all future rewards count equally. `G_t = Σ R_{t+k+1}` (undiscounted sum).
- **Is γ = 1 safe for a continuing task?** ❌ No — the return becomes an infinite sum that may diverge unless rewards are bounded and the series converges.

In [ ]:
rewards_ex1 = [-0.1] * 10 + [10.0]

G_myopic   = compute_returns(rewards_ex1, gamma=0.0)
G_farsight = compute_returns(rewards_ex1, gamma=1.0)

print(f'γ=0 (myopic)      → G_0 = {G_myopic[0]:.2f}  (only R_1 = -0.1 counts)')
print(f'γ=1 (far-sighted) → G_0 = {G_farsight[0]:.2f}   (10×(-0.1) + 10 = 9.0)')
print()
print('γ=1 is UNSAFE for continuing tasks: the sum diverges when rewards never stop.')

### Exercise 2 — Minimum γ for far reward to contribute ≥ 50% of G_0

With rewards = [1, 1, …, 1, **100**] (100 at step 50), we find the smallest γ
such that the discounted far reward `γ^50 × 100` is at least half of `G_0`.

In [ ]:
rewards_ex2 = [1.0] * 50 + [100.0]

print('γ      G_0      far-reward contrib   fraction')
print('-' * 50)
threshold_gamma = None
for gamma in np.arange(0.50, 1.001, 0.01):
    G = compute_returns(rewards_ex2, gamma)
    far_contrib = (gamma ** 50) * 100.0
    fraction    = far_contrib / G[0]
    print(f'γ={gamma:.2f}  G_0={G[0]:7.2f}  far={far_contrib:6.2f}  frac={fraction:.3f}')
    if fraction >= 0.50 and threshold_gamma is None:
        threshold_gamma = gamma
        print(f'  ^^^ THRESHOLD CROSSED ^^^')
    if gamma > threshold_gamma + 0.05 if threshold_gamma else False:
        break

print(f'\n✅ Minimum γ ≈ {threshold_gamma:.2f} for the far reward to contribute ≥ 50% of G_0')

### Exercise 3 (Challenge) — `compute_returns_forward`

**Key idea:** define a *gamma-weighted suffix sum*

$$w_t = \gamma^t \cdot R_{t+1}, \qquad S_t = \sum_{j=t}^{T-1} w_j$$

Then $G_t = S_t / \gamma^t$ — no reversal needed; `S` is built with `np.cumsum` on the reversed `w` array.

In [ ]:
def compute_returns_forward(rewards, gamma):
    """
    Compute discounted returns G_t *without* reversing the array.

    Trick: factor out gamma^t from the geometric series.
      G_t = sum_{k=0}^{T-t-1} gamma^k * R_{t+k+1}
          = (1/gamma^t) * sum_{j=t}^{T-1} gamma^j * R_{j+1}
          = S_t / gamma^t
    where S_t is a suffix-sum over the gamma-weighted reward array w[j] = gamma^j * R[j+1].
    numpy's cumsum on the reversed w gives all suffix sums in O(n).
    """
    n = len(rewards)
    t_idx = np.arange(n)
    w = (gamma ** t_idx) * np.asarray(rewards, dtype=float)   # gamma^t * R_t
    S = np.cumsum(w[::-1])[::-1]                               # suffix sums
    G = np.where(t_idx == 0, S, S / (gamma ** t_idx))         # avoid 0^0 edge case
    return G


# ── Verification ──────────────────────────────────────────────
np.random.seed(42)
rewards_test = np.random.randn(20)
gamma_test   = 0.9

G_ref = compute_returns(rewards_test, gamma_test)          # original (reversed)
G_fwd = compute_returns_forward(rewards_test, gamma_test)  # new (forward)

print('t     G_ref      G_fwd      diff')
for t in range(len(G_ref)):
    print(f'{t:2d}  {G_ref[t]:9.5f}  {G_fwd[t]:9.5f}  {abs(G_ref[t]-G_fwd[t]):.2e}')

print(f'\nMax absolute difference: {np.max(np.abs(G_ref - G_fwd)):.2e}')
assert np.allclose(G_ref, G_fwd)
print('✅ compute_returns_forward matches compute_returns (backwards)!')